In [1]:
import numpy as np
import pandas as pd
import os
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from pandas import json_normalize

In [2]:
# --- Configuration ---
DATA_DIR = './data'
# Ensure the data directory exists
os.makedirs(DATA_DIR, exist_ok=True)

# Define file names
X_TRAIN_FILE = "X_train_processed.npy"
X_KAGGLE_FILE = "X_kaggle_processed.npy"
Y_TRAIN_FILE =  "y_train.npy"

In [3]:
# --- 1. Load Data ---
def load_data():
    """Loads the processed feature arrays and the target labels."""
    try:
        X_train_full = np.load(os.path.join(DATA_DIR, X_TRAIN_FILE))
        X_kaggle_full = np.load(os.path.join(DATA_DIR, X_KAGGLE_FILE))
        y_full = np.load(os.path.join(DATA_DIR, Y_TRAIN_FILE))

        print(f"X_train_full loaded: {X_train_full.shape}")
        print(f"X_kaggle_full loaded: {X_kaggle_full.shape}")
        print(f"y_full loaded: {y_full.shape}")

        return X_train_full, X_kaggle_full, y_full

    except FileNotFoundError as e:
        print(f"Error: Could not find required file. Please check DATA_DIR and file names.")
        print(f"Missing file: {e}")
        return None, None, None

X_train_full, X_kaggle_full, y_full = load_data()

if X_train_full is None:
    # Exit if data loading failed
    exit()

# Split the full training data into training and a small validation set
# We use this validation set for a final test after Grid Search
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_full, test_size=0.1, random_state=42, stratify=y_full
)

print("-" * 50)
print(f"Training set size: {X_train.shape[0]}")
print(f"Validation set size: {X_val.shape[0]}")
print("-" * 50)

X_train_full loaded: (154914, 776)
X_kaggle_full loaded: (103380, 776)
y_full loaded: (154914,)
--------------------------------------------------
Training set size: 139422
Validation set size: 15492
--------------------------------------------------


In [4]:
# --- 2. Define Model and Parameter Grid for Cross-Validation ---

# For robustness, we'll use a Pipeline that first scales the data
# (important for Logistic Regression) and then applies the model.
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('logreg', LogisticRegression(random_state=42, solver='liblinear'))
])

# Define the grid of hyperparameters to search over
param_grid = {
    # C is the inverse of regularization strength (smaller C means stronger regularization)
    'logreg__C': [0.01, 0.1, 10],
    # Penalty type: L1 regularization can lead to sparser models (useful for feature selection)
    'logreg__penalty': ['l2']
}

# Setup GridSearchCV for cross-validation
# We use 5-fold cross-validation on the main training set (X_train, y_train)
grid_search = GridSearchCV(
    pipeline,
    param_grid,
    cv=5,                 # 5-fold cross-validation
    scoring='accuracy',   # Metric to optimize
    n_jobs=-1,            # Use all available cores
    verbose=1
)

In [5]:

# --- 3. Train Model and Find Best Parameters ---
print("Starting Grid Search Cross-Validation...")
grid_search.fit(X_train, y_train)

# Output the best parameters found
best_params = grid_search.best_params_
print("\n" + "=" * 50)
print("✅ Grid Search Completed.")
print(f"Best cross-validation score (Accuracy): {grid_search.best_score_:.4f}")
print(f"Best Parameters: {best_params}")
print("=" * 50)

Starting Grid Search Cross-Validation...
Fitting 5 folds for each of 3 candidates, totalling 15 fits



✅ Grid Search Completed.
Best cross-validation score (Accuracy): 0.7938
Best Parameters: {'logreg__C': 10, 'logreg__penalty': 'l2'}


In [9]:

# --- 4. Evaluate Final Model on Validation Set ---
# The best estimator is the model trained on the full X_train set using the best parameters.
best_model = grid_search.best_estimator_

# Make predictions on the held-out validation set
y_val_pred = best_model.predict(X_val)

# Evaluate performance
val_accuracy = accuracy_score(y_val, y_val_pred)
print(f"\nModel Performance on Held-out Validation Set:")
print(f"Accuracy: {val_accuracy:.4f}")
print("\nClassification Report:")
print(classification_report(y_val, y_val_pred))


# --- 5. Infer Model on X_kaggle_processed ---
# Apply the best model to the final dataset for submission
print("-" * 50)
print("Starting inference on X_kaggle_processed...")

# The best_model (a Pipeline) automatically handles scaling for the Kaggle data
kaggle_predictions = best_model.predict(X_kaggle_full)

print(f"Inference complete. Generated {len(kaggle_predictions)} predictions.")
print(f"Example predictions (first 10): {kaggle_predictions[:10]}")
print("-" * 50)

# --- 6. Build Kaggle submission JSON ---
print("Building submission.csv ...")

# Load challenge IDs from test.csv
X_kaggle = pd.read_json("data/kaggle_test.jsonl", lines=True)
X_kaggle = json_normalize(X_kaggle.to_dict(orient="records"))
challenge_ids = X_kaggle["challenge_id"].values

assert len(challenge_ids) == len(kaggle_predictions), \
    "Mismatch: predictions and challenge_ids lengths differ!"

submission = pd.DataFrame([
    {
        "ID": int(cid),
        "Prediction": pred
    }
    for cid, pred in zip(challenge_ids, kaggle_predictions)
])

# Save to CSV
submission.to_csv("./submission/submission.csv", index=False)

print("✅ submission.csv created successfully!")


Model Performance on Held-out Validation Set:
Accuracy: 0.7911

Classification Report:
              precision    recall  f1-score   support

           0       0.76      0.88      0.82      8268
           1       0.83      0.69      0.75      7224

    accuracy                           0.79     15492
   macro avg       0.80      0.78      0.79     15492
weighted avg       0.80      0.79      0.79     15492

--------------------------------------------------
Starting inference on X_kaggle_processed...
Inference complete. Generated 103380 predictions.
Example predictions (first 10): [1 1 0 0 0 0 0 0 0 0]
--------------------------------------------------
Building submission.csv ...
✅ submission.csv created successfully!
